In [ ]:
# 📦 Install if not already done
# !pip install langchain openai faiss-cpu tiktoken python-dotenv

# 1️⃣ Imports
from langchain_ollama import ChatOllama
from langchain_classic.tools import Tool
from langchain_classic.agents import initialize_agent, AgentType
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings
from langchain_classic.chains import RetrievalQA
from langchain_classic.memory import ConversationBufferMemory


# 3️⃣ Setup LLM
llm = ChatOllama(
    model='mistral',
    temperature=0)

# 4️⃣ Create Vector DB (Retriever)

with open("sample.txt", "r", encoding="utf-8") as f:
    text_data = f.read()

# 🧠 Split the text into smaller chunks
from langchain_text_splitters import CharacterTextSplitter
splitter = CharacterTextSplitter(separator="\n", chunk_size=300, chunk_overlap=50)
texts = splitter.split_text(text_data)
embedding = OllamaEmbeddings(model='nomic-embed-text')
vectorstore = FAISS.from_texts(texts, embedding)
retriever = vectorstore.as_retriever()

# 5️⃣ Create RetrievalQA Chain
qa_chain = RetrievalQA.from_chain_type(llm=llm, retriever=retriever)

# 6️⃣ Convert to LangChain Tool
qa_tool = Tool(
    name="LangChainRetriever",
    func=qa_chain.run,
    description="Use this to answer questions about LangChain framework, features, or its creator."
)

# 7️⃣ Setup Memory (to support conversational context)
memory = ConversationBufferMemory(memory_key="chat_history")

# 8️⃣ Initialize Agent with Tool + Memory
agent = initialize_agent(
    tools=[qa_tool],
    llm=llm,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,  # ReAct agent
    memory=memory,
    verbose=True,
    handle_parsing_errors=True
)

# 9️⃣ Ask Questions (RAG-Style)
print("1️⃣ First Question")
res1 = agent.invoke("What is LangChain?")
print("Answer:", res1)

print("\n2️⃣ Follow-up")
res2 = agent.invoke("Who created it?")
print("Answer:", res2)

print("\n3️⃣ Combined Reasoning")
res3 = agent.invoke("Explain LangChain's use in AI workflows.")
print("Answer:", res3)


1️⃣ First Question


> Entering new AgentExecutor chain...
 Thought: Do I need to use a tool? Yes
Action: LangChainRetriever
Action Input: What is LangChain?
Observation:  LangChain is a framework designed for building applications using Large Language Models (LLMs). It was created by Harrison Chase. The framework supports features such as RAG (Reflect, Act, Generate), agents, memory, tools, among others. It's commonly used in chatbots, document Q&A, and AI workflow.
Thought: Do I need to use a tool? Yes
Action: LangChainRetriever
Action Input: What is LangChain?
Observation:  LangChain is a framework designed for building applications using Large Language Models (LLMs). It was created by Harrison Chase. The framework supports features such as RAG (Reflect, Act, Generate), agents, memory, tools, among others. It's commonly used in chatbots, document Q&A, and AI workflow.
Thought: Do I need to use a tool? No
AI: LangChain is a framework designed for building applications using Large Lan